# Maharashtra Soybean Yield Prediction: Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on historical agricultural crop statistics, growing-season agroclimatology (NASA POWER), and topsoil pedological properties (ISRIC SoilGrids 2.0) for **Soybean (*Glycine max*)** during the **Kharif season** across agricultural districts in Maharashtra, India (1990–2017).

### Objectives
1. Inspect dataset dimensions, schema, and completeness.
2. Analyze statewide and district-level historical soybean yield distributions and temporal trends.
3. Evaluate weather patterns (precipitation totals, temperature extremes, solar radiation, dry spells).
4. Examine soil property distributions (pH, clay, sand, organic carbon, cation exchange capacity).
5. Investigate bivariate correlations between candidate predictor features and crop yield.
6. Differentiate empirical statistical associations from causal claims.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 10

# Locate processed dataset relative to project root
data_path = os.path.abspath(os.path.join('..', 'data', 'processed', 'crop_yield_maharashtra.csv'))

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f'Processed dataset not found at {data_path}. '
        'Please execute python -m app.data.pipeline --stage all to run data acquisition and alignment.'
    )

df = pd.read_csv(data_path)
print(f'Successfully loaded aligned dataset: {df.shape[0]} rows x {df.shape[1]} columns.')

## 1. Dataset Dimensions and Schema Audit

In [ ]:
print('--- Column Data Types and Null Audit ---')
print(df.info())
display(df.head(5))

## 2. Target Variable Analysis: Soybean Yield (t/ha)
Soybean yield in tonnes per hectare (t/ha) derived from official reported production and cultivated area.

In [ ]:
yield_stats = df['yield_t_ha'].describe()
print('--- Soybean Yield Summary Statistics (t/ha) ---')
print(yield_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['yield_t_ha'], kde=True, ax=axes[0], color='#2e7d32', bins=25)
axes[0].set_title('Distribution of Historical Soybean Yield (t/ha)')
axes[0].set_xlabel('Yield (t/ha)')

sns.boxplot(y=df['yield_t_ha'], ax=axes[1], color='#81c784')
axes[1].set_title('Soybean Yield Boxplot and Outliers')
axes[1].set_ylabel('Yield (t/ha)')
plt.tight_layout()
plt.show()

## 3. Historical Temporal Trends (1990-2017)
Observing statewide annual mean yield and identifying known meteorological shock years (e.g., severe monsoon deficits in 2004, 2014, and 2015).

In [ ]:
yearly_trends = df.groupby('year').agg({
    'yield_t_ha': ['mean', 'std', 'min', 'max'],
    'cumulative_rainfall_mm': 'mean'
}).reset_index()
yearly_trends.columns = ['year', 'mean_yield', 'std_yield', 'min_yield', 'max_yield', 'mean_rainfall']

fig, ax1 = plt.subplots(figsize=(13, 6))
color = '#1b5e20'
ax1.plot(yearly_trends['year'], yearly_trends['mean_yield'], color=color, marker='o', linewidth=2.5, label='Mean Yield (t/ha)')
ax1.fill_between(yearly_trends['year'], yearly_trends['mean_yield'] - yearly_trends['std_yield'], yearly_trends['mean_yield'] + yearly_trends['std_yield'], color=color, alpha=0.15, label='+/- 1 Std Dev')
ax1.set_xlabel('Harvest Year', fontweight='bold')
ax1.set_ylabel('Soybean Yield (t/ha)', color=color, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color2 = '#1565c0'
ax2.bar(yearly_trends['year'], yearly_trends['mean_rainfall'], alpha=0.25, color=color2, width=0.6, label='Kharif Rainfall (mm)')
ax2.set_ylabel('Mean Kharif Rainfall (mm)', color=color2, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Maharashtra Soybean Yield Trends vs. Growing Season Precipitation (1990-2017)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. District-Level Yield Variability
Comparison across Maharashtra agricultural districts, spanning Vidarbha, Marathwada, and Western Maharashtra.

In [ ]:
dist_order = df.groupby('district')['yield_t_ha'].median().sort_values(ascending=False).index
plt.figure(figsize=(14, 7))
sns.boxplot(data=df, x='district', y='yield_t_ha', order=dist_order, palette='Greens_r')
plt.xticks(rotation=45, ha='right')
plt.title('Historical Soybean Yield Distributions by District (Ordered by Median)', fontsize=12, fontweight='bold')
plt.xlabel('District')
plt.ylabel('Yield (t/ha)')
plt.tight_layout()
plt.show()

## 5. Predictor Features: Weather and Soil Distributions

In [ ]:
weather_cols = [
    'cumulative_rainfall_mm', 'mean_temp_max_c', 'mean_temp_min_c',
    'mean_solar_radiation_mj_m2', 'mean_relative_humidity_pct', 'consecutive_dry_days_max'
]
soil_cols = [
    'soil_ph', 'organic_carbon_pct', 'clay_pct', 'sand_pct', 'bulk_density_g_cm3', 'cation_exchange_capacity_cmol_kg'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(weather_cols):
    ax = axes[i // 3, i % 3]
    sns.histplot(df[col], kde=True, ax=ax, color='#0288d1')
    ax.set_title(col)
plt.suptitle('Distributions of Seasonal Weather Predictors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(soil_cols):
    ax = axes[i // 3, i % 3]
    sns.histplot(df[col], kde=True, ax=ax, color='#8d6e63')
    ax.set_title(col)
plt.suptitle('Distributions of Topsoil Properties', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis and Feature Associations with Yield

In [ ]:
numeric_features = weather_cols + soil_cols + ['yield_t_ha']
corr_matrix = df[numeric_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1, square=True)
plt.title('Pearson Correlation Matrix: Crop Yield, Weather and Soil Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

yield_corr = corr_matrix['yield_t_ha'].drop('yield_t_ha').sort_values(ascending=False)
print('--- Feature Correlations with Soybean Yield (t/ha) ---')
print(yield_corr)

## 7. Critical Discussion: Correlation vs. Causation

> **Important Scientific Caution**:
> 1. **Statistical Association != Direct Agronomic Causality**: For instance, while higher clay content (Vertisols) strongly correlates with higher soybean yields in rainfed systems due to moisture retention during terminal dry spells, excessive clay without drainage can induce waterlogging during intense monsoon bursts.
> 2. **Unobserved Confounders**: The historical dataset does not record farm-level management factors such as certified seed usage, sowing dates, pest infestations (e.g. Spodoptera litura), and fertilizer application rates.
> 3. **Macro-Level Spatial Resolution**: Meteorological values represent 0.5 degree gridded district averages and SoilGrids values represent 250m gridded topsoil estimates, which provide macro-level regional decision support rather than farm-parcel guarantees.